# Resumo

## Análise Exploratória de Dataset

Este notebook realiza uma análise exploratória completa do dataset TON_IoT, focando em:
- Caracterização inicial do dataset
- Análise de qualidade dos dados
- Estatísticas descritivas
- Distribuição de classes
- Análise de features e correlações

# Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3
import warnings
import os
from datetime import datetime
from tqdm import tqdm
warnings.filterwarnings('ignore')

# Visualization settings
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10
sns.set_palette("husl")

# Seed for reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Inputs

In [ ]:
table_name = "CIC_APT_IIoT_2024"

# Data Loading

In [ ]:
# Connect to SQLite database
db_path = '../data/sqlite/data.db'
conn = sqlite3.connect(db_path)

# Check available tables
cursor = conn.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()
print("Available tables in the database:")
for table in tables:
    print(f"  • {table[0]}")

# Análise

## 1. Caracterização Inicial do Dataset

In [ ]:
# Helper function to escape column names for SQL
def escape_column_name(col_name):
    return f'"{col_name}"'

In [ ]:
print("=" * 60)
print("INITIAL DATASET CHARACTERIZATION")
print("=" * 60)

print(f"\n📊 Dataset Dimensions:")
print(f"   • Rows: {total_rows:,}")
print(f"   • Columns: {len(column_names)}")

print(f"\n📋 Data Types:")
type_counts = {}
for col_type in column_types.values():
    type_counts[col_type] = type_counts.get(col_type, 0) + 1
for dtype, count in type_counts.items():
    print(f"   • {dtype}: {count} columns")

print(f"\n📝 Column Names:")
print(f"   Total: {len(column_names)} features")
for i, col in enumerate(column_names, 1):
    print(f"   {i:2d}. {col}")

## 2. Análise de Qualidade dos Dados

In [ ]:
# Missing values analysis using SQL
print("=" * 80)
print("DATA QUALITY ANALYSIS")
print("=" * 80)
print("\n📊 Analyzing data quality...")

quality_analysis = []

for col in tqdm(column_names, desc="Analyzing data quality", unit="column"):
    
    # Count NULL values
    cursor.execute(f"SELECT COUNT(*) FROM {table_name} WHERE {col} IS NULL")
    null_count = cursor.fetchone()[0]
    
    # Count empty strings
    cursor.execute(f"SELECT COUNT(*) FROM {table_name} WHERE TRIM({col}) = ''")
    empty_count = cursor.fetchone()[0]
    
    # Count unique values
    cursor.execute(f"SELECT COUNT(DISTINCT {col}) FROM {table_name}")
    unique_count = cursor.fetchone()[0]
    
    total_missing = null_count + empty_count
    missing_percent = (total_missing / total_rows * 100) if total_rows > 0 else 0
    
    quality_analysis.append({
        'Column': col,
        'Type': column_types.get(col, 'TEXT'),
        'Unique_Values': unique_count,
        'Missing_Values': total_missing,
        'Missing_Percent': round(missing_percent, 2),
        'Completeness_%': round(100 - missing_percent, 2)
    })

quality_df = pd.DataFrame(quality_analysis).sort_values('Missing_Percent', ascending=False)

print(f"\n📊 General Summary:")
columns_with_missing = (quality_df['Missing_Values'] > 0).sum()
total_missing = quality_df['Missing_Values'].sum()
avg_completeness = quality_df['Completeness_%'].mean()

print(f"   • Columns with missing values: {columns_with_missing}")
print(f"   • Total missing values: {total_missing:,}")
print(f"   • Average completeness percentage: {avg_completeness:.2f}%")

print(f"\n📋 Details by Column (showing first 20):")
pd.set_option('display.max_rows', 20)
print(quality_df.to_string(index=False))
pd.reset_option('display.max_rows')
if len(quality_df) > 20:
    print(f"\n... and {len(quality_df) - 20} more columns")

In [ ]:
# Missing values visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Chart 1: Missing values per column
missing_data = quality_df[quality_df['Missing_Values'] > 0].sort_values('Missing_Values', ascending=True)
if len(missing_data) > 0:
    axes[0].barh(range(len(missing_data)), missing_data['Missing_Percent'], color='coral')
    axes[0].set_yticks(range(len(missing_data)))
    axes[0].set_yticklabels(missing_data['Column'], fontsize=8)
    axes[0].set_xlabel('Missing Values Percentage (%)', fontsize=12)
    axes[0].set_title('Missing Values per Column', fontsize=14, fontweight='bold')
    axes[0].grid(axis='x', alpha=0.3)
else:
    axes[0].text(0.5, 0.5, 'No missing values found', 
                 ha='center', va='center', fontsize=14, transform=axes[0].transAxes)
    axes[0].set_title('Missing Values per Column', fontsize=14, fontweight='bold')

# Chart 2: Overall completeness
completeness = quality_df['Completeness_%'].values
axes[1].hist(completeness, bins=30, color='steelblue', edgecolor='black', alpha=0.7)
axes[1].axvline(completeness.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {completeness.mean():.2f}%')
axes[1].set_xlabel('Completeness (%)', fontsize=12)
axes[1].set_ylabel('Number of Columns', fontsize=12)
axes[1].set_title('Column Completeness Distribution', fontsize=14, fontweight='bold')
axes[1].legend()
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Duplicate detection using SQL
print("=" * 60)
print("DUPLICATE ANALYSIS")
print("=" * 60)
print("   Analyzing duplicates (this may take a while for large datasets)...")

# Count total rows
cursor.execute(f"SELECT COUNT(*) FROM {table_name}")
total_count = cursor.fetchone()[0]

# Count distinct rows (comparing all columns)
# This is expensive but necessary for exact duplicate detection
all_cols = ', '.join([escape_column_name(col) for col in column_names])
cursor.execute(f"SELECT COUNT(*) FROM (SELECT DISTINCT {all_cols} FROM {table_name})")
unique_count = cursor.fetchone()[0]

duplicates_count = total_count - unique_count
duplicates_percent = (duplicates_count / total_count * 100) if total_count > 0 else 0

print(f"   • Total records: {total_count:,}")
print(f"   • Unique records: {unique_count:,}")
print(f"   • Duplicate records: {duplicates_count:,}")
print(f"   • Duplicate percentage: {duplicates_percent:.2f}%")

if duplicates_count > 0:
    print(f"\n⚠️  Warning: {duplicates_percent:.2f}% of records are duplicates")
else:
    print(f"\n✓ No duplicates found")

## 3. Estatísticas Descritivas

In [ ]:
# Classify columns based on SQLite types and sample data
print("=" * 60)
print("DESCRIPTIVE STATISTICS")
print("=" * 60)
print("\n📊 Classifying features...")

# Sample a small subset to determine data types
sample_query = f"SELECT * FROM {table_name} LIMIT 1000"
df_sample = pd.read_sql_query(sample_query, conn)

numeric_cols = df_sample.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df_sample.select_dtypes(include=['object']).columns.tolist()

# Also check SQLite types
sqlite_numeric_types = ['INTEGER', 'REAL', 'NUMERIC']
sqlite_numeric_cols = [col for col in column_names if column_types.get(col, '').upper() in sqlite_numeric_types]

# Combine both approaches
all_numeric = list(set(numeric_cols + sqlite_numeric_cols))
all_categorical = [col for col in column_names if col not in all_numeric]

# Store for later use
numeric_cols = all_numeric
categorical_cols = all_categorical

print(f"\n📊 Feature Classification:")
print(f"   • Numeric: {len(numeric_cols)}")
print(f"   • Categorical: {len(categorical_cols)}")

if numeric_cols:
    print(f"\n📈 Numeric Features:")
    for i, col in enumerate(numeric_cols, 1):
        print(f"   {i:2d}. {col}")
    
if categorical_cols:
    print(f"\n📝 Categorical Features:")
    for i, col in enumerate(categorical_cols[:20], 1):  # Show first 20
        print(f"   {i:2d}. {col}")
    if len(categorical_cols) > 20:
        print(f"   ... and {len(categorical_cols) - 20} more categorical features")

In [ ]:
# Descriptive statistics for numeric features
if numeric_cols:
    print("\n" + "=" * 80)
    print("DESCRIPTIVE STATISTICS - NUMERIC FEATURES")
    print("=" * 80)
    
    desc_stats = df[numeric_cols].describe()
    desc_stats.loc['var'] = df[numeric_cols].var()
    desc_stats.loc['skew'] = df[numeric_cols].skew()
    desc_stats.loc['kurtosis'] = df[numeric_cols].kurtosis()
    
    print(desc_stats.round(2).to_string())

In [ ]:
# Descriptive statistics for categorical features using SQL
if categorical_cols:
    print("\n" + "=" * 80)
    print("DESCRIPTIVE STATISTICS - CATEGORICAL FEATURES")
    print("=" * 80)
    
    cat_stats = []
    for col in tqdm(categorical_cols, desc="Computing categorical statistics", unit="column"):
        col_escaped = escape_column_name(col)
        
        # Get unique count
        cursor.execute(f"SELECT COUNT(DISTINCT {col_escaped}) FROM {table_name}")
        unique_count = cursor.fetchone()[0]
        
        # Get mode (most frequent value) and its frequency
        mode_query = f"""
        SELECT {col_escaped} as value, COUNT(*) as count
        FROM {table_name}
        WHERE {col_escaped} IS NOT NULL AND TRIM({col_escaped}) != ''
        GROUP BY {col_escaped}
        ORDER BY count DESC
        LIMIT 1
        """
        cursor.execute(mode_query)
        mode_result = cursor.fetchone()
        
        if mode_result:
            mode_value = mode_result[0]
            mode_frequency = mode_result[1]
            mode_percent = (mode_frequency / total_rows * 100) if total_rows > 0 else 0
        else:
            mode_value = 'N/A'
            mode_frequency = 0
            mode_percent = 0
        
        cat_stats.append({
            'Column': col,
            'Unique_Values': unique_count,
            'Mode': mode_value,
            'Mode_Frequency': mode_frequency,
            'Mode_Percent': round(mode_percent, 2)
        })
    
    cat_stats_df = pd.DataFrame(cat_stats)
    print(cat_stats_df.to_string(index=False))

In [ ]:
# Visualization of distributions of main numeric features
if numeric_cols:
    # Select up to 12 most important numeric features for visualization
    important_numeric = numeric_cols[:12] if len(numeric_cols) > 12 else numeric_cols
    
    n_cols = 3
    n_rows = (len(important_numeric) + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 6*n_rows))
    axes = axes.flatten() if n_rows > 1 else [axes] if n_cols == 1 else axes
    
    for idx, col in enumerate(important_numeric):
        ax = axes[idx]
        data = df[col].dropna()
        
        if len(data) > 0:
            # Histogram with KDE
            ax.hist(data, bins=50, alpha=0.7, color='steelblue', edgecolor='black', density=True)
            
            # Add density line
            try:
                from scipy.stats import gaussian_kde
                if len(data) > 1:
                    kde = gaussian_kde(data)
                    x_range = np.linspace(data.min(), data.max(), 200)
                    ax.plot(x_range, kde(x_range), 'r-', linewidth=2, label='KDE')
            except:
                pass
            
            ax.set_title(f'{col}', fontsize=11, fontweight='bold')
            ax.set_xlabel('Value', fontsize=10)
            ax.set_ylabel('Density', fontsize=10)
            ax.legend()
            ax.grid(alpha=0.3)
        else:
            ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
            ax.set_title(f'{col}', fontsize=11, fontweight='bold')
    
    # Hide extra axes
    for idx in range(len(important_numeric), len(axes)):
        axes[idx].axis('off')
    
    plt.suptitle('Numeric Features Distributions', fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

In [ ]:
# Boxplots to detect outliers
if numeric_cols:
    important_numeric = numeric_cols[:12] if len(numeric_cols) > 12 else numeric_cols
    
    n_cols = 3
    n_rows = (len(important_numeric) + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 6*n_rows))
    axes = axes.flatten() if n_rows > 1 else [axes] if n_cols == 1 else axes
    
    for idx, col in enumerate(important_numeric):
        ax = axes[idx]
        data = df[col].dropna()
        
        if len(data) > 0:
            bp = ax.boxplot(data, vert=True, patch_artist=True)
            bp['boxes'][0].set_facecolor('lightblue')
            bp['boxes'][0].set_alpha(0.7)
            
            ax.set_title(f'{col}', fontsize=11, fontweight='bold')
            ax.set_ylabel('Value', fontsize=10)
            ax.grid(axis='y', alpha=0.3)
        else:
            ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
            ax.set_title(f'{col}', fontsize=11, fontweight='bold')
    
    for idx in range(len(important_numeric), len(axes)):
        axes[idx].axis('off')
    
    plt.suptitle('Boxplots - Outlier Detection', fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

In [ ]:
# Outlier detection using IQR
if numeric_cols:
    print("\n" + "=" * 80)
    print("OUTLIER DETECTION (IQR Method)")
    print("=" * 80)
    
    outliers_summary = []
    for col in numeric_cols:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        
        outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)][col]
        outliers_count = len(outliers)
        outliers_percent = (outliers_count / len(df)) * 100
        
        outliers_summary.append({
            'Feature': col,
            'Q1': Q1,
            'Q3': Q3,
            'IQR': IQR,
            'Outliers': outliers_count,
            'Outliers_Percent': outliers_percent
        })
    
    outliers_df = pd.DataFrame(outliers_summary).sort_values('Outliers', ascending=False)
    print(outliers_df.to_string(index=False))

## 5. Análise de Distribuição de Classes

In [ ]:
# Check if classification columns exist
class_columns = []
if 'label' in column_names:
    class_columns.append('label')
if 'type' in column_names:
    class_columns.append('type')

print("=" * 80)
print("CLASS DISTRIBUTION ANALYSIS")
print("=" * 80)

if class_columns:
    for col in class_columns:
        print(f"\n📊 Distribution of column '{col}':")
        print("-" * 80)
        
        col_escaped = escape_column_name(col)
        
        # Get class distribution using SQL
        dist_query = f"""
        SELECT {col_escaped} as class, COUNT(*) as count
        FROM {table_name}
        GROUP BY {col_escaped}
        ORDER BY count DESC
        """
        cursor.execute(dist_query)
        results = cursor.fetchall()
        
        if results:
            classes = [r[0] for r in results]
            counts = [r[1] for r in results]
            percents = [(c / total_rows * 100) for c in counts]
            
            distribution_df = pd.DataFrame({
                'Class': classes,
                'Count': counts,
                'Percent': [round(p, 2) for p in percents]
            })
            
            print(distribution_df.to_string(index=False))
            
            # Balance
            max_percent = max(percents)
            min_percent = min(percents)
            imbalance_ratio = max_percent / min_percent if min_percent > 0 else float('inf')
            
            print(f"\n   • Total classes: {len(classes)}")
            print(f"   • Most frequent class: {classes[0]} ({max_percent:.2f}%)")
            print(f"   • Least frequent class: {classes[-1]} ({min_percent:.2f}%)")
            print(f"   • Imbalance ratio: {imbalance_ratio:.2f}:1")
            
            if imbalance_ratio > 10:
                print(f"   ⚠️  Highly imbalanced dataset!")
            elif imbalance_ratio > 5:
                print(f"   ⚠️  Moderately imbalanced dataset")
            else:
                print(f"   ✓ Relatively balanced dataset")
        else:
            print("   No data found")
else:
    print("\n⚠️  No classification column found (label or type)")

In [ ]:
# Class distribution visualization
if class_columns:
    n_cols = len(class_columns)
    fig, axes = plt.subplots(1, n_cols, figsize=(8*n_cols, 6))
    
    if n_cols == 1:
        axes = [axes]
    
    for idx, col in enumerate(class_columns):
        ax = axes[idx]
        col_escaped = escape_column_name(col)
        
        # Get distribution from SQL
        dist_query = f"""
        SELECT {col_escaped} as class, COUNT(*) as count
        FROM {table_name}
        GROUP BY {col_escaped}
        ORDER BY count DESC
        LIMIT 20
        """
        dist_df = pd.read_sql_query(dist_query, conn)
        
        if len(dist_df) > 0:
            classes = dist_df['class'].values
            counts = dist_df['count'].values
            percents = (counts / total_rows * 100)
            
            # Bar chart
            bars = ax.bar(range(len(classes)), counts, 
                         color=sns.color_palette("husl", len(classes)), 
                         edgecolor='black', linewidth=0.7)
            
            # Add values on bars
            for i, (bar, val, pct) in enumerate(zip(bars, counts, percents)):
                height = bar.get_height()
                ax.text(val, bar.get_y() + bar.get_height()/2,
                       f'{val:,}\n({pct:.1f}%)',
                       ha='center', va='bottom', fontsize=9, fontweight='bold')
            
            ax.set_xticks(range(len(classes)))
            ax.set_xticklabels([str(c) for c in classes], rotation=45, ha='right')
            ax.set_ylabel('Frequency', fontsize=12)
            ax.set_title(f'Class Distribution - {col}', fontsize=14, fontweight='bold')
            ax.grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Pie chart for class distribution
if class_columns:
    n_cols = len(class_columns)
    fig, axes = plt.subplots(1, n_cols, figsize=(8*n_cols, 6))
    
    if n_cols == 1:
        axes = [axes]
    
    for idx, col in enumerate(class_columns):
        ax = axes[idx]
        col_escaped = escape_column_name(col)
        
        # Get distribution from SQL
        dist_query = f"""
        SELECT {col_escaped} as class, COUNT(*) as count
        FROM {table_name}
        GROUP BY {col_escaped}
        ORDER BY count DESC
        LIMIT 20
        """
        dist_df = pd.read_sql_query(dist_query, conn)
        
        if len(dist_df) > 0:
            classes = dist_df['class'].values
            counts = dist_df['count'].values
            percents = (counts / total_rows * 100)
            
            colors = sns.color_palette("husl", len(classes))
            wedges, texts, autotexts = ax.pie(counts, 
                                              labels=[str(c) for c in classes],
                                              autopct='%1.1f%%',
                                              colors=colors,
                                              startangle=90,
                                              textprops={'fontsize': 10})
            
            # Improve readability
            for autotext in autotexts:
                autotext.set_color('white')
                autotext.set_fontweight('bold')
                autotext.set_fontsize(9)
            
            ax.set_title(f'Percentage Distribution - {col}', fontsize=14, fontweight='bold', pad=20)
    
    plt.tight_layout()
    plt.show()

## 6. Análise de Features e Correlações

In [ ]:
# Correlation matrix for numeric features
if len(numeric_cols) > 1:
    print("=" * 80)
    print("CORRELATION ANALYSIS BETWEEN NUMERIC FEATURES")
    print("=" * 80)
    
    # Calculate correlation
    correlation_matrix = df[numeric_cols].corr()
    
    # Identify strong correlations (|r| > 0.7)
    strong_correlations = []
    for i in range(len(correlation_matrix.columns)):
        for j in range(i+1, len(correlation_matrix.columns)):
            corr_value = correlation_matrix.iloc[i, j]
            if abs(corr_value) > 0.7:
                strong_correlations.append({
                    'Feature_1': correlation_matrix.columns[i],
                    'Feature_2': correlation_matrix.columns[j],
                    'Correlation': corr_value
                })
    
    if strong_correlations:
        print(f"\n🔗 Strong Correlations (|r| > 0.7): {len(strong_correlations)} pairs found")
        strong_corr_df = pd.DataFrame(strong_correlations).sort_values('Correlation', key=abs, ascending=False)
        print(strong_corr_df.to_string(index=False))
    else:
        print("\n✓ No strong correlations found (|r| > 0.7)")
        strong_corr_df = pd.DataFrame(columns=['Feature_1', 'Feature_2', 'Correlation'])
    
    # Correlation matrix statistics
    corr_values = correlation_matrix.values
    # Remove diagonal (values 1.0)
    mask = ~np.eye(corr_values.shape[0], dtype=bool)
    corr_off_diag = corr_values[mask]
    
    print(f"\n📊 Correlation Statistics:")
    print(f"   • Mean: {corr_off_diag.mean():.3f}")
    print(f"   • Standard Deviation: {corr_off_diag.std():.3f}")
    print(f"   • Minimum: {corr_off_diag.min():.3f}")
    print(f"   • Maximum: {corr_off_diag.max():.3f}")
else:
    print("⚠️  Less than 2 numeric features found for correlation analysis")

In [ ]:
# Correlation heatmap
if len(numeric_cols) > 1:
    # Limit to 20 features for better visualization
    cols_to_plot = numeric_cols[:20] if len(numeric_cols) > 20 else numeric_cols
    
    fig, ax = plt.subplots(figsize=(14, 12))
    
    corr_subset = df[cols_to_plot].corr()
    
    # Create mask for upper diagonal
    mask = np.triu(np.ones_like(corr_subset, dtype=bool))
    
    sns.heatmap(corr_subset, 
                mask=mask,
                annot=True, 
                fmt='.2f', 
                cmap='coolwarm', 
                center=0,
                square=True, 
                linewidths=0.5,
                cbar_kws={"shrink": 0.8},
                annot_kws={'fontsize': 8},
                ax=ax)
    
    ax.set_title('Correlation Matrix - Numeric Features', fontsize=16, fontweight='bold', pad=20)
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

In [ ]:
# Variance analysis of numeric features
if numeric_cols:
    print("\n" + "=" * 80)
    print("FEATURE VARIANCE ANALYSIS")
    print("=" * 80)
    
    variance_analysis = []
    for col in numeric_cols:
        data = df[col].dropna()
        if len(data) > 0:
            variance_analysis.append({
                'Feature': col,
                'Variance': data.var(),
                'Std_Deviation': data.std(),
                'Coefficient_Variation': (data.std() / data.mean()) if data.mean() != 0 else np.inf,
                'Range': data.max() - data.min()
            })
    
    variance_df = pd.DataFrame(variance_analysis).sort_values('Variance', ascending=False)
    
    # Identify features with low variance (potentially less informative)
    low_variance_threshold = variance_df['Variance'].quantile(0.1)
    low_variance_features = variance_df[variance_df['Variance'] < low_variance_threshold]
    
    print(f"\n📊 Top 10 Features with Highest Variance:")
    print(variance_df.head(10).to_string(index=False))
    
    if len(low_variance_features) > 0:
        print(f"\n⚠️  Features with Low Variance (potentially less informative):")
        print(low_variance_features.to_string(index=False))
    else:
        print(f"\n✓ All features show significant variance")

In [ ]:
# Cardinality analysis of categorical features using SQL
if categorical_cols:
    print("\n" + "=" * 80)
    print("CARDINALITY ANALYSIS - CATEGORICAL FEATURES")
    print("=" * 80)
    
    cardinality_analysis = []
    for col in tqdm(categorical_cols, desc="Analyzing cardinality", unit="column"):
        col_escaped = escape_column_name(col)
        
        # Get unique count
        cursor.execute(f"SELECT COUNT(DISTINCT {col_escaped}) FROM {table_name}")
        unique_count = cursor.fetchone()[0]
        
        # Get total non-null count
        cursor.execute(f"SELECT COUNT(*) FROM {table_name} WHERE {col_escaped} IS NOT NULL AND TRIM({col_escaped}) != ''")
        total_count = cursor.fetchone()[0]
        
        cardinality_ratio = unique_count / total_count if total_count > 0 else 0
        
        cardinality_analysis.append({
            'Feature': col,
            'Unique_Values': unique_count,
            'Total_Records': total_count,
            'Cardinality_Ratio': cardinality_ratio,
            'Category': 'High' if cardinality_ratio > 0.5 else 'Medium' if cardinality_ratio > 0.1 else 'Low'
        })
    
    cardinality_df = pd.DataFrame(cardinality_analysis).sort_values('Unique_Values', ascending=False)
    
    print(cardinality_df.to_string(index=False))
    
    # Identify features with high cardinality (potentially problematic)
    high_cardinality = cardinality_df[cardinality_df['Cardinality_Ratio'] > 0.9]
    if len(high_cardinality) > 0:
        print(f"\n⚠️  Features with High Cardinality (>90% unique values):")
        print(high_cardinality[['Feature', 'Unique_Values', 'Cardinality_Ratio']].to_string(index=False))

In [ ]:
# Visualization of strongest correlations
if len(numeric_cols) > 1 and 'strong_corr_df' in locals() and len(strong_corr_df) > 0:
    # Select top 10 strongest correlations
    top_correlations = strong_corr_df.head(10) if len(strong_corr_df) >= 10 else strong_corr_df
    
    fig, ax = plt.subplots(figsize=(12, 8))
    
    y_pos = np.arange(len(top_correlations))
    colors = ['red' if x < 0 else 'blue' for x in top_correlations['Correlation'].values]
    
    bars = ax.barh(y_pos, top_correlations['Correlation'].values, color=colors, alpha=0.7, edgecolor='black')
    
    # Add values on bars
    for i, (bar, val) in enumerate(zip(bars, top_correlations['Correlation'].values)):
        ax.text(val, bar.get_y() + bar.get_height()/2,
               f'{val:.3f}',
               ha='left' if val > 0 else 'right',
               va='center', fontsize=9, fontweight='bold')
    
    # Labels
    labels = [f"{row['Feature_1']} ↔ {row['Feature_2']}" 
              for _, row in top_correlations.iterrows()]
    ax.set_yticks(y_pos)
    ax.set_yticklabels(labels, fontsize=9)
    ax.set_xlabel('Correlation (r)', fontsize=12)
    ax.set_title('Top Strong Correlations between Features', fontsize=14, fontweight='bold')
    ax.axvline(0, color='black', linestyle='-', linewidth=0.5)
    ax.grid(axis='x', alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## Resumo Executivo

Este notebook apresentou uma análise exploratória completa do dataset TON_IoT, cobrindo:

1. ✅ **Caracterização Inicial**: Dimensões, tipos de dados e uso de memória
2. ✅ **Qualidade dos Dados**: Valores faltantes, duplicatas e completude
3. ✅ **Estatísticas Descritivas**: Análise de features numéricas e categóricas
4. ✅ **Distribuição de Classes**: Balanceamento e frequências
5. ✅ **Análise de Features**: Correlações, variância e cardinalidade

### Próximos Passos Sugeridos:
- Análise temporal (se timestamp disponível)
- Análise de protocolos e serviços de rede
- Feature engineering baseado nos insights obtidos
- Preparação para modelagem (preprocessing)

# Gerando MD

In [ ]:
# Create results directory if it doesn't exist
results_dir = '../results/dataset_analysis'
os.makedirs(results_dir, exist_ok=True)

# Generate markdown report using variables collected throughout the notebook
md_content = []
md_content.append("# Dataset Analysis Report\n")
md_content.append(f"## Exploratory Dataset Analysis\n\n")
md_content.append(f"This report presents a complete exploratory analysis of the {table_name} dataset.\n\n")
md_content.append(f"**Analysis Date**: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
md_content.append(f"**Dataset**: {table_name}\n")
md_content.append(f"**Sample Size**: {total_rows:,} records\n\n")
md_content.append("---\n\n")

# 1. Initial Dataset Characterization
md_content.append("## 1. Initial Dataset Characterization\n\n")
md_content.append("### Dataset Dimensions\n")
md_content.append(f"- **Rows**: {total_rows:,}\n")
md_content.append(f"- **Columns**: {len(column_names):,}\n\n")

# Get actual database size (if not already calculated)
if 'db_size_mb' not in locals():
    db_size_bytes = os.path.getsize(db_path)
    db_size_mb = db_size_bytes / (1024**2)
    cursor.execute("PRAGMA page_count")
    page_count = cursor.fetchone()[0]
    cursor.execute("PRAGMA page_size")
    page_size = cursor.fetchone()[0]
else:
    # Recalculate db_size_bytes if db_size_mb exists
    db_size_bytes = db_size_mb * (1024**2)

md_content.append("### Database Storage Size\n")
md_content.append(f"- **Total database size**: {db_size_mb:.2f} MB\n")
md_content.append(f"- **Average size per row**: ~{(db_size_bytes / total_rows) if total_rows > 0 else 0:.2f} bytes\n\n")

# Data types from column_types dictionary
type_counts = {}
for col_type in column_types.values():
    type_counts[col_type] = type_counts.get(col_type, 0) + 1
md_content.append("### Data Types\n")
for dtype, count in type_counts.items():
    md_content.append(f"- **{dtype}**: {count} columns\n")
md_content.append("\n")

md_content.append("### Column Names\n")
md_content.append(f"Total: {len(column_names)} features\n\n")
for i, col in enumerate(column_names, 1):
    md_content.append(f"{i}. {col}\n")
md_content.append("\n---\n\n")

# 2. Data Quality Analysis (using quality_df already calculated)
md_content.append("## 2. Data Quality Analysis\n\n")
md_content.append("### General Summary\n")
columns_with_missing = (quality_df['Missing_Values'] > 0).sum()
total_missing = quality_df['Missing_Values'].sum()
avg_completeness = quality_df['Completeness_%'].mean()

md_content.append(f"- **Columns with missing values**: {columns_with_missing}\n")
md_content.append(f"- **Total missing values**: {total_missing:,}\n")
md_content.append(f"- **Average completeness percentage**: {avg_completeness:.2f}%\n\n")

md_content.append("### Duplicate Analysis\n\n")
# Use duplicates_count already calculated in Cell 15
if 'duplicates_count' not in locals():
    # Recalculate if not available
    cursor.execute(f"SELECT COUNT(*) FROM {table_name}")
    total_count = cursor.fetchone()[0]
    all_cols = ', '.join([escape_column_name(col) for col in column_names])
    cursor.execute(f"SELECT COUNT(*) FROM (SELECT DISTINCT {all_cols} FROM {table_name})")
    unique_count = cursor.fetchone()[0]
    duplicates_count = total_count - unique_count
    duplicates_percent = (duplicates_count / total_count * 100) if total_count > 0 else 0
else:
    duplicates_percent = (duplicates_count / total_rows * 100) if total_rows > 0 else 0

md_content.append(f"- **Duplicate records**: {duplicates_count:,}\n")
md_content.append(f"- **Duplicate percentage**: {duplicates_percent:.2f}%\n")
md_content.append(f"- **Unique records**: {total_rows - duplicates_count:,}\n\n")
if duplicates_count > 0:
    md_content.append(f"⚠️ **Warning**: {duplicates_percent:.2f}% of records are duplicates\n\n")
else:
    md_content.append("✓ **No duplicates found**\n\n")
md_content.append("---\n\n")

# 3. Descriptive Statistics
md_content.append("## 3. Descriptive Statistics\n\n")
md_content.append("### Feature Classification\n")
md_content.append(f"- **Numeric**: {len(numeric_cols) if 'numeric_cols' in locals() else 0}\n")
md_content.append(f"- **Categorical**: {len(categorical_cols) if 'categorical_cols' in locals() else 0}\n\n")

# Use cat_stats_df if available, otherwise use quality_df information
if 'categorical_cols' in locals() and categorical_cols:
    md_content.append("### Descriptive Statistics - Categorical Features\n\n")
    if 'cat_stats_df' in locals():
        # Use already calculated cat_stats_df
        md_content.append("| Column | Unique_Values | Mode | Mode_Frequency | Mode_Percent |\n")
        md_content.append("|--------|---------------|------|----------------|-------------|\n")
        for _, row in cat_stats_df.head(20).iterrows():
            md_content.append(f"| {row['Column']} | {row['Unique_Values']} | {row['Mode']} | {row['Mode_Frequency']:,} | {row['Mode_Percent']:.2f}% |\n")
        if len(cat_stats_df) > 20:
            md_content.append(f"\n*Showing top 20 of {len(cat_stats_df)} categorical features*\n")
    else:
        # Use quality_df for basic info
        md_content.append("| Column | Unique_Values | Completeness_% |\n")
        md_content.append("|--------|---------------|----------------|\n")
        for _, row in quality_df[quality_df['Column'].isin(categorical_cols)].head(20).iterrows():
            md_content.append(f"| {row['Column']} | {row['Unique_Values']} | {row['Completeness_%']:.2f}% |\n")
    md_content.append("\n---\n\n")

# 4. Class Distribution Analysis (using SQLite)
md_content.append("## 4. Class Distribution Analysis\n\n")
if 'class_columns' in locals() and class_columns:
    for col in class_columns:
        md_content.append(f"### Distribution of column '{col}'\n\n")
        col_escaped = escape_column_name(col)
        
        # Get class distribution using SQL
        dist_query = f"""
        SELECT {col_escaped} as class, COUNT(*) as count
        FROM {table_name}
        GROUP BY {col_escaped}
        ORDER BY count DESC
        """
        cursor.execute(dist_query)
        results = cursor.fetchall()
        
        if results:
            classes = [r[0] for r in results]
            counts = [r[1] for r in results]
            percents = [(c / total_rows * 100) for c in counts]
            
            md_content.append("| Class | Count | Percent |\n")
            md_content.append("|-------|-------|----------|\n")
            for cls, count, pct in zip(classes, counts, percents):
                md_content.append(f"| {cls} | {count:,} | {pct:.2f}% |\n")
            md_content.append("\n")
            
            max_percent = max(percents)
            min_percent = min(percents)
            imbalance_ratio = max_percent / min_percent if min_percent > 0 else float('inf')
            
            md_content.append("**Summary:**\n")
            md_content.append(f"- **Total classes**: {len(classes)}\n")
            md_content.append(f"- **Most frequent class**: {classes[0]} ({max_percent:.2f}%)\n")
            md_content.append(f"- **Least frequent class**: {classes[-1]} ({min_percent:.2f}%)\n")
            md_content.append(f"- **Imbalance ratio**: {imbalance_ratio:.2f}:1\n\n")
            
            if imbalance_ratio > 10:
                md_content.append("⚠️ **Highly imbalanced dataset!**\n\n")
            elif imbalance_ratio > 5:
                md_content.append("⚠️ **Moderately imbalanced dataset**\n\n")
            else:
                md_content.append("✓ **Relatively balanced dataset**\n\n")
        else:
            md_content.append("No data found\n\n")
else:
    md_content.append("⚠️ **No classification column found (label or type)**\n\n")
md_content.append("---\n\n")

# 5. Feature Analysis
md_content.append("## 5. Feature Analysis and Correlations\n\n")
if 'numeric_cols' in locals() and len(numeric_cols) > 1:
    md_content.append("⚠️ **Correlation analysis requires loading numeric data into memory**\n\n")
    md_content.append("**Note**: Since all features are stored as strings in SQLite, correlation analysis requires type conversion to numeric format first. This would require loading data into memory.\n\n")
else:
    md_content.append("⚠️ **Less than 2 numeric features found for correlation analysis**\n\n")

# Cardinality Analysis (using cardinality_df if available, otherwise use quality_df)
if 'categorical_cols' in locals() and categorical_cols:
    md_content.append("### Cardinality Analysis - Categorical Features\n\n")
    if 'cardinality_df' in locals():
        # Use already calculated cardinality_df
        high_cardinality = cardinality_df[cardinality_df['Cardinality_Ratio'] > 0.9]
        if len(high_cardinality) > 0:
            md_content.append("**Features with High Cardinality (>90% unique values):**\n\n")
            md_content.append("| Feature | Unique_Values | Cardinality_Ratio |\n")
            md_content.append("|---------|---------------|-------------------|\n")
            for _, row in high_cardinality.iterrows():
                md_content.append(f"| {row['Feature']} | {row['Unique_Values']} | {row['Cardinality_Ratio']:.6f} |\n")
            md_content.append("\n")
        
        md_content.append("**Cardinality Categories:**\n")
        high_count = len(cardinality_df[cardinality_df['Category'] == 'High'])
        medium_count = len(cardinality_df[cardinality_df['Category'] == 'Medium'])
        low_count = len(cardinality_df[cardinality_df['Category'] == 'Low'])
        md_content.append(f"- **High** (>50% unique): {high_count} features\n")
        md_content.append(f"- **Medium** (10-50% unique): {medium_count} features\n")
        md_content.append(f"- **Low** (<10% unique): {low_count} features\n\n")
    else:
        # Use quality_df to estimate cardinality
        quality_cat = quality_df[quality_df['Column'].isin(categorical_cols)].copy()
        quality_cat['Cardinality_Ratio'] = quality_cat['Unique_Values'] / total_rows
        quality_cat['Category'] = quality_cat['Cardinality_Ratio'].apply(
            lambda x: 'High' if x > 0.5 else 'Medium' if x > 0.1 else 'Low'
        )
        high_cardinality = quality_cat[quality_cat['Cardinality_Ratio'] > 0.9]
        if len(high_cardinality) > 0:
            md_content.append("**Features with High Cardinality (>90% unique values):**\n\n")
            md_content.append("| Feature | Unique_Values | Cardinality_Ratio |\n")
            md_content.append("|---------|---------------|-------------------|\n")
            for _, row in high_cardinality.iterrows():
                md_content.append(f"| {row['Column']} | {row['Unique_Values']} | {row['Cardinality_Ratio']:.6f} |\n")
            md_content.append("\n")
md_content.append("---\n\n")

# Executive Summary
md_content.append("## 6. Executive Summary\n\n")
md_content.append("This analysis presented a complete exploratory analysis of the dataset, covering:\n\n")
md_content.append("1. ✅ **Initial Characterization**: Dimensions, data types and database storage size\n")
md_content.append("2. ✅ **Data Quality**: Missing values, duplicates and completeness\n")
md_content.append("3. ✅ **Descriptive Statistics**: Analysis of numeric and categorical features\n")
md_content.append("4. ✅ **Class Distribution**: Balance and frequencies\n")
md_content.append("5. ✅ **Feature Analysis**: Cardinality and variance\n\n")

md_content.append("### Key Findings\n\n")
md_content.append(f"1. **Data Quality**: {'Excellent' if total_missing == 0 and duplicates_count == 0 else 'Needs attention'} - {avg_completeness:.2f}% completeness, {total_missing:,} missing values, {duplicates_count:,} duplicates\n")
md_content.append(f"2. **Data Types**: {len(type_counts)} unique data types - {len(categorical_cols) if 'categorical_cols' in locals() else 0} categorical, {len(numeric_cols) if 'numeric_cols' in locals() else 0} numeric\n")
if 'class_columns' in locals() and class_columns:
    # Count classes from SQL
    for col in class_columns:
        col_escaped = escape_column_name(col)
        cursor.execute(f"SELECT COUNT(DISTINCT {col_escaped}) FROM {table_name}")
        num_classes = cursor.fetchone()[0]
        md_content.append(f"3. **Class Distribution**: {num_classes} classes found in '{col}'\n")
        break
    else:
        md_content.append("3. **Class Distribution**: No classification columns found\n")
else:
    md_content.append("3. **Class Distribution**: No classification columns found\n")
if 'high_cardinality' in locals():
    md_content.append(f"4. **High Cardinality**: {len(high_cardinality)} features with >90% unique values\n")
else:
    md_content.append("4. **High Cardinality**: Analysis not performed\n")
md_content.append("\n### Suggested Next Steps\n\n")
md_content.append("1. **Data Type Conversion**: Convert numeric features from string to appropriate numeric types\n")
md_content.append("2. **Feature Engineering**: Remove constant features, consider dimensionality reduction for high-cardinality features\n")
md_content.append("3. **Data Loading**: Investigate if additional data needs to be loaded\n")
md_content.append("4. **Temporal Analysis**: Analyze patterns over time using timestamp features\n")
md_content.append("5. **Preprocessing**: Prepare data for modeling with appropriate encoding strategies\n")
md_content.append("\n---\n\n")

md_content.append("## Appendix: Dataset Information\n\n")
md_content.append(f"- **Dataset**: {table_name}\n")
md_content.append(f"- **Sample Size**: {total_rows:,} records\n")
md_content.append(f"- **Total Features**: {len(column_names)}\n")
md_content.append(f"- **Database Size**: {db_size_mb:.2f} MB\n")
md_content.append(f"- **Analysis Date**: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
md_content.append(f"- **Database**: SQLite (`{db_path}`)\n\n")
md_content.append("---\n\n")
md_content.append("*Report generated from dataset_analysis.ipynb notebook*\n")

# Write to file
output_file = os.path.join(results_dir, f'{table_name}_analysis_report.md')
with open(output_file, 'w', encoding='utf-8') as f:
    f.writelines(md_content)

print(f"✓ Markdown report generated successfully!")
print(f"  Location: {output_file}")
print(f"  Size: {len(''.join(md_content)):,} characters")